# M6 · 组合优化与质量门

**学习目标：** 在**你的** W7900 上测量组合 BEST 配置（Turbo 蒸馏 8 步 + INT8 权重）与 B0 的差距，产生成对质量证据，并让最终判定留给人工。已发表的 6.12× 只是参考；你自己的数字才是你的。

参考讲义：[M06 讲义](../course/M06_combined_quality/LESSON.zh-CN.md) · [English](./07_combined_and_quality.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse explicitly selected physical runs (ids shown + classified)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="07")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse earlier physical runs to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing explicitly selected physical run(s) "
          "instead of launching new ones (metered-GPU mode; provenance shown)")
    return False

## 1 · 物理运行 B0 与 BEST
两个臂都按你的模式运行；复用 M1 的 B0 时会显式标明 run_id。

In [ ]:
def run_m6_pair():
    prof = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    b0 = session.run("B0", profile=prof)
    best = session.run("BEST", profile=prof)
    return b0, best

m6 = gpu_cell(run_m6_pair) if want_own_arms() else None
m6_origin = "CURRENT_SESSION_PHYSICAL"
if m6 is None:
    ids = [session.latest_campaign_run("B0"), session.latest_campaign_run("BEST")]
    if all(ids):
        m6 = (session.consume(ids[0], why="explicit reuse (shown run_id): B0"),
              session.consume(ids[1], why="explicit reuse (shown run_id): BEST"))
        m6_origin = session.consumed[-1].get("category", "COMMITTED_PHYSICAL_REFERENCE")
if m6 is not None:
    ab = rt.ab_compare(m6[0], m6[1], label="B0 vs BEST (turbo+int8)")
    print({k: ab[k] for k in ("classification", "speedup_median",
                              "latency_delta_s", "vram_delta_mb")})
    _stamp(_origin_stamp(m6_origin), f"B0/BEST speedup {ab['speedup_median']}x "
          f"[{ab['classification']}] vs frozen reference 6.12x")
    fig = rt.plot_ab(ab) or rt.plot_ab(ab, metric="vram")
    if fig: fig.show()

## 2 · 质量维度 —— 成对证据，人工判定
组合变更（模型变体 × 调度 × 精度）采用**距离语义**。生成或检查成对网格与原生分辨率海报裁剪；用三级评分标准打分。这里的 LPIPS 是用于*分流*评审的感知距离——它永远不下结论。

> **质量判定永远由人完成。** 任何单元都不会自动批准图像质量结论。历史所有者判定**只**适用于其原始评估的部署范围；新的物理运行需要自己的评审（在此之前保持 `PENDING_HUMAN_REVIEW`）。

In [ ]:
print("three-tier rubric:  (1) no perceptible change  "
      "(2) slight degradation, still deployable  (3) unacceptable")
print()
def run_own_quality():
    session.run_quality("BEST", reference="B0")
    qdir = ROOT / "runs/quality/BEST"
    for f in sorted(qdir.glob("*.png")):
        print("  your artifact:", f.name)
    import json as _j
    lp = qdir / "lpips_table.json"
    if lp.is_file():
        print(_j.dumps(_j.loads(lp.read_text()), indent=1)[:800])
    return qdir

q = gpu_cell(run_own_quality)
if q is None:
    print("frozen campaign artifacts (reference):")
    for f in sorted((ROOT / "evidence/quality/BEST").glob("*")):
        print("  ", f.name)
    print("historical M6 BEST independent human review: PENDING_HUMAN_REVIEW "
          "(owner decision boundary preserved)")
_stamp("GATE STATUS", "your verdict on YOUR images is PENDING_HUMAN_REVIEW "
      "until you complete the rubric on the grid above")

## 3 · 诚实解读你的组合结果
1. 你测得的 B0/BEST 比值接近 6.12× 吗？若不接近，是哪个臂与参考不同（分别对比 B0 与 BEST 中位数和 55.206 秒 / 9.023 秒）？
2. 组合比值不等于 F1×F2×L1 相乘——为什么？
3. BEST 的 VRAM 峰值比 B0 低约 6.6 GB——这能带来什么？

In [ ]:
if m6 is not None:
    b0s, bests = m6[0].stats(), m6[1].stats()
    print(f"B0   {b0s['median_s']:7.2f}s (ref 55.206)  VRAM {b0s['vram_device_peak_mb']:.0f}MB (ref 33153)")
    print(f"BEST {bests['median_s']:7.2f}s (ref 9.023)  VRAM {bests['vram_device_peak_mb']:.0f}MB (ref 26514)")
    print("report the delta per arm, not only the ratio")

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment BEST --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。